##SCD Type 1

nb_sales_silver_gold_v1: is the SCD Type 1

nb_sales_silver_gold_v1

In [0]:
%sql
MERGE INTO
  intellibicatalogo.intellibi_gold.refinedMonthlySales AS TGT
USING (
  SELECT
    *
  FROM
    intellibicatalogo.intellibi_silver.cleanedMonthlySales
  WHERE
    ingest_ts
      > (
        SELECT
          COALESCE(MAX(last_u_ts), TO_TIMESTAMP('1900-01-01'), 'YYYY-MM-DD')
        FROM
          intellibicatalogo.intellibi_gold.refinedMonthlySales
      )
) AS SRC
ON
  TGT.Customer_ID = SRC.Customer_ID
WHEN MATCHED AND
  (
    TGT.Customer_Name <> SRC.Customer_Name
    OR TGT.Gender <> SRC.Gender
    OR TGT.Age <> SRC.Age
    OR TGT.City <> SRC.City
    OR TGT.custpurPrice <> SRC.custpurPrice
    OR TGT.purchase_date <> SRC.purchase_date
  )
  THEN UPDATE SET
  TGT.Customer_Name = SRC.Customer_Name,
  TGT.Gender = SRC.Gender,
  TGT.Age = SRC.Age,
  TGT.City = SRC.City,
  TGT.custpurPrice = SRC.custpurPrice,
  TGT.purchase_date = SRC.purchase_date,
  TGT.last_u_ts = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    TGT.Customer_ID,
    TGT.Customer_Name,
    TGT.Gender,
    TGT.Age,
    TGT.City,
    TGT.custpurPrice,
    TGT.purchase_date,
    TGT.intial_load_ts,
    TGT.last_u_ts
  )
  VALUES (
    SRC.Customer_ID,
    SRC.Customer_Name,
    SRC.Gender,
    SRC.Age,
    SRC.City,
    SRC.custpurPrice,
    SRC.purchase_date,
    current_timestamp(),
    current_timestamp()
  );

In [0]:
%sql
SELECT * FROM intellibicatalogo.intellibi_gold.refinedMonthlySales;